# SQL Worksheet — Week2

Use the following tables from the Riva Data Platform:

- `rivadataplatform.dataproduct.dim_batch`
- `rivadataplatform.dataproduct.dim_class`
- `rivadataplatform.dataproduct.fact_attendance`
- `rivadataplatform.dataproduct.dim_student`
- `rivadataplatform.dataproduct.dim_date`

**Instructions**
- Write SQL for each question.
- Do not modify the source data.
- Use clear aliases where JOINs are involved.
- Unless a question specifically asks for a particular column, select only the columns needed to answer it.


## Tables / Relationships

Useful keys:
- `dim_student.student_key` ↔ `fact_attendance.student_key`
- `dim_class.class_key` ↔ `fact_attendance.class_key`
- `dim_batch.batch_key` ↔ `fact_attendance.batch_key`
- `dim_class.batch_id` ↔ `dim_batch.batch_id`

## Question 1 — Attendance by Student Location
Build the query in these steps:

**1.1 — Select student location**
From `dim_student`, select the city and replace null city values with `Unknown city`.

**1.2 — Join attendance**
Join `fact_attendance` using `student_key` and add the attendance record identifier and status.

**1.3 — Add class and batch dimensions**
Join `dim_class` using `class_key` and `dim_batch` using `batch_key`.

**1.4 — Group and aggregate**
Group by city and batch name. Count distinct students, total attendance records, and `Late` or `Absent` issue records.

**1.5 — Filter and sort**
Keep only groups with at least one issue and order by issue records descending, then city.

Use readable labels for null batch names.

In [0]:
--Write your code here
select 
coalesce(ds.city,'Unknown city') as city,
coalesce(db.batch_name, 'Unknown batch') as batch_name,
count(fa.attendance_id) as total_attendanc2_records,
count(distinct ds.student_id) as total_students,
sum(case when fa.attendance_status in ('Late','Absent') then 1 else 0 end) as issue_records

from rivadataplatform.dataproduct.dim_student ds
join rivadataplatform.dataproduct.fact_attendance fa
on ds.student_key = fa.student_key
left join rivadataplatform.dataproduct.dim_class dc
on dc.class_key = fa.class_key
left join rivadataplatform.dataproduct.dim_batch db
on fa.batch_key = db.batch_key

group by coalesce(ds.city,'Unknown city'), coalesce(db.batch_name, 'Unknown batch')
having sum(case when fa.attendance_status in ('Late','Absent') then 1 else 0 end) > 0
order by issue_records desc, city;

In [0]:
--Write your code here

## Question 2 — Class Calendar Status Counts
Build the query in these steps:

**2.1 — Select class calendar columns**
From `dim_class`, select class date, class day, topic, and class key. Replace null topics with `Topic not assigned`.

**2.2 — Join attendance**
Join `fact_attendance` using `class_key` and add the attendance record identifier and status.

**2.3 — Add batch and date details**
Join `dim_batch` using `batch_key` and `dim_date` using `date_key`. Use the date dimension day name when available.

**2.4 — Group and count statuses**
Group by class date, day name, topic, and batch name. Count `Present`, `Late`, and `Absent` records.

**2.5 — Filter and sort**
Keep classes with attendance records and order by class date.

In [0]:
--Write your code here
SELECT dc.class_date,
COALESCE(dd.day_name,dc.class_day) AS day_name,
COALESCE(dc.topic,'Topic not assigned') AS Topic_Name, 
dc.class_key,
count(fa.attendance_id) AS attendance_records,
SUM(CASE WHEN fa.attendance_status='LATE' THEN 1 ELSE 0 END) AS late_count,
SUM(CASE WHEN fa.attendance_status='ABSENT' THEN 1 ELSE 0 END) AS absent_count,
SUM(CASE WHEN fa.attendance_status='PRESENT' THEN 1 ELSE 0 END) AS present_count,
db.batch_name

FROM rivadataplatform.dataproduct.dim_class AS dc
JOIN rivadataplatform.dataproduct.fact_attendance AS fa
ON fa.class_key = dc.class_key
LEFT JOIN rivadataplatform.dataproduct.dim_batch AS db
ON db.batch_key = fa.batch_key
LEFT JOIN rivadataplatform.dataproduct.dim_date AS dd
ON dd.date_key = fa.date_key

GROUP BY dc.class_date, COALESCE(dd.day_name,dc.class_day), COALESCE(dc.topic,'Topic not assigned'), dc.class_key, db.batch_name
HAVING attendance_records > 0
ORDER BY dc.class_date

In [0]:
--Write your code here

## Question 3 — Null Profile and Attendance Check
Build the query in these steps:

**3.1 — Select student profile columns**
From `dim_student`, select student ID, student name, and phone number. Label blank or null phone numbers as `Phone missing`.

**3.2 — Add attendance**
Use a `LEFT JOIN` to connect `fact_attendance` through `student_key` and select the attendance record identifier.

**3.3 — Add class topics**
Use a `LEFT JOIN` to connect `dim_class` through `class_key` and inspect the topic.

**3.4 — Group and aggregate**
Group by student and phone status. Count attendance records and records with a missing topic.

**3.5 — Filter and sort**
Keep students with a missing phone or at least one missing topic, then order by student name.

In [0]:
--Write your code here
SELECT ds.student_id, ds.student_name,
COALESCE(ds.phone_no,'Phone Missing') As phone_no,
SUM(CASE WHEN dc.topic IS NULL OR dc.topic = '' THEN 1 ELSE 0 END) AS missing_topics,
COUNT(fa.attendance_count) AS attendance_records

FROM rivadataplatform.dataproduct.dim_student AS ds
LEFT JOIN rivadataplatform.dataproduct.fact_attendance fa 
ON fa.student_key = ds.student_key
LEFT JOIN rivadataplatform.dataproduct.dim_class dc
ON dc.class_key = fa.class_key

GROUP BY ds.student_id, ds.student_name,COALESCE(ds.phone_no,'Phone Missing')
HAVING missing_topics > 0 OR nullif(phone_no,'') IS NULL
ORDER BY ds.student_name

In [0]:
--Write your code here

## Question 4 — Batch Attendance Rate
Build the query in these steps:

**4.1 — Select batch columns**
From `dim_batch`, select `batch_id` and `batch_name`.

**4.2 — Join attendance**
Join `fact_attendance` using `batch_key` and add the attendance record identifier, student key, and status.

**4.3 — Group and count**
Group by batch ID and batch name. Count total records and distinct students.

**4.4 — Calculate the rate**
Count `Present` and `Late` as attended, divide by total records, multiply by 100, and use `NULLIF` to avoid division by zero.

**4.5 — Filter and sort**
Keep batches with at least one `Absent` record and order by attendance rate descending.

In [0]:
--Write your code here
select db.batch_id,
db.batch_name,
COUNT(fa.attendance_id) AS total_records,
COUNT(distinct fa.student_key) AS total_students,
(SUM(CASE WHEN fa.attendance_status = 'Present' OR fa.attendance_status = 'Late' THEN 1 ELSE 0 END) * 100) / NULLIF(COUNT(fa.attendance_id),0) AS attendance_rate 

FROM rivadataplatform.dataproduct.dim_batch db
JOIN rivadataplatform.dataproduct.fact_attendance fa
ON fa.batch_key = db.batch_key

GROUP BY db.batch_id, db.batch_name
HAVING SUM(CASE WHEN fa.attendance_status = 'Absent' THEN 1 ELSE 0 END) > 0
Order By attendance_rate

In [0]:
--Write your code here

## Question 5 — Repeated Attendance by Student and Topic
Build the query in these steps:

**5.1 — Select student columns**
From `dim_student`, select student ID and student name.

**5.2 — Join attendance and classes**
Join `fact_attendance` using `student_key`, then join `dim_class` using `class_key`. Add the class date and topic.

**5.3 — Normalize the topic**
Replace null topic values with `Topic not assigned` so they form one grouping value.

**5.4 — Group and aggregate**
Group by student and topic. Count records, find the first and last class dates, and count `Late` or `Absent` records.

**5.5 — Filter and sort**
Keep student-topic groups with more than one record and order by attendance records descending, then student name.

In [0]:
--Write your code here

SELECT ds.student_id, ds.student_name,
coalesce(dc.topic,'Topic not assigned') as topic, 
COUNT(fa.attendance_id) AS attendance_records,
MIN(dc.class_date) AS first_class_date,
MAX (dc.class_date) AS last_class_date,
SUM(CASE WHEN fa.attendance_status = 'Late' OR fa.attendance_status = 'Absent' THEN 1 ELSE 0 END) as issue_records

FROM rivadataplatform.dataproduct.dim_student ds
JOIN rivadataplatform.dataproduct.fact_attendance fa
ON fa.student_key = ds.student_key
JOIN rivadataplatform.dataproduct.dim_class dc
ON dc.class_key = fa.class_key

GROUP BY ds.student_id, ds.student_name, topic
HAVING attendance_records > 0
ORDER BY attendance_records desc, ds.student_name

In [0]:
--Write your code here

## Question 6 — Student Present Coverage
Build the query in these steps:

**6.1 — Select student columns**
From `dim_student`, select student ID and student name.

**6.2 — Join attendance and batch**
Use `LEFT JOIN` to add `fact_attendance` through `student_key` and `dim_batch` through `batch_key`.

**6.3 — Group by student and batch**
Group by student ID, student name, and a null-safe batch label.

**6.4 — Calculate coverage metrics**
Count total attendance records, distinct classes, and `Present` records.

**6.5 — Sort the result**
Include students with zero attendance and order by present count descending, then student name.

In [0]:
--Write your code here
SELECT ds.student_id, ds.student_name,
COALESCE(db.batch_name, 'No attendance batch') AS batch_name,
COUNT(fa.attendance_id) AS attendance_records,
COUNT(DISTINCT fa.class_key) AS distinct_classes,
SUM(CASE WHEN fa.attendance_status = 'Present' THEN 1 ELSE 0 END) AS present_count

FROM rivadataplatform.dataproduct.dim_student ds
LEFT JOIN rivadataplatform.dataproduct.fact_attendance fa
ON fa.student_key = ds.student_key
LEFT JOIN rivadataplatform.dataproduct.dim_batch db
ON fa.batch_key = db.batch_key

GROUP BY ds.student_id, ds.student_name,  COALESCE(db.batch_name, 'No attendance batch')
ORDER BY present_count DESC, ds.student_name;

In [0]:
--Write your code here